In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('drilling_operations.db')


In [2]:
## Check total records in each table
sql = """
-- 1. Record counts for all tables
SELECT 'PAC'                AS table_name, COUNT(*) AS record_count FROM PAC
UNION ALL
SELECT 'Region',               COUNT(*) FROM Region
UNION ALL
SELECT 'Field',                COUNT(*) FROM Field
UNION ALL
SELECT 'Rig',                  COUNT(*) FROM Rig
UNION ALL
SELECT 'Well',                 COUNT(*) FROM Well
UNION ALL
SELECT 'AFE',                  COUNT(*) FROM AFE
UNION ALL
SELECT 'WellNpt',              COUNT(*) FROM WellNpt
UNION ALL
SELECT 'WellCompletionCost',   COUNT(*) FROM WellCompletionCost
UNION ALL
SELECT 'WellDrilling',         COUNT(*) FROM WellDrilling
UNION ALL
SELECT 'Report',               COUNT(*) FROM Report;
"""
df_counts = pd.read_sql_query(sql, conn)
print("=== Record Counts ===")
print(df_counts.to_string(index=False))

=== Record Counts ===
        table_name  record_count
               PAC            14
            Region            18
             Field            32
               Rig            29
              Well            64
               AFE            64
           WellNpt            64
WellCompletionCost             0
      WellDrilling             0
            Report           132


In [3]:
## Verify referential integrity
sql = """
SELECT w.WellName, w.FieldName
FROM   Well w
LEFT JOIN Field f ON w.FieldName = f.FieldName
WHERE  f.FieldName IS NULL;
"""
df_integrity = pd.read_sql_query(sql, conn)
print("\n=== Wells with No Matching Field (expect 0 rows) ===")
print(df_integrity.to_string(index=False) if not df_integrity.empty else "✓ No issues found")


=== Wells with No Matching Field (expect 0 rows) ===
✓ No issues found


In [4]:
## Check for data quality
sql = """
SELECT
    COUNT(*)                        AS total_afe_records,
    COUNT(DISTINCT a.WellName)      AS unique_wells,
    COUNT(DISTINCT w.RigName)       AS unique_rigs,
    ROUND(AVG(a.FinalCost), 2)      AS avg_final_cost,
    ROUND(AVG(a.FinalDays), 2)      AS avg_final_days
FROM AFE a
JOIN Well w ON w.WellName = a.WellName;
"""
df_quality = pd.read_sql_query(sql, conn)
print("\n=== Data Quality Summary ===")
print(df_quality.to_string(index=False))


=== Data Quality Summary ===
 total_afe_records  unique_wells  unique_rigs  avg_final_cost  avg_final_days
                64            64           28     11466525.26           33.03
